<a href="https://colab.research.google.com/github/Suman18-bit/Deep-Learning/blob/main/ANN_with_Data_Visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Importing the necessary libraries
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Fetching the California Housing dataset
housing = fetch_california_housing()

# Splitting the data into training, validation, and test sets
X_train_full, X_test, y_train_full, y_test = train_test_split(housing.data, housing.target, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, random_state=42)

# Standardizing the data using StandardScaler
scaler = StandardScaler()

# Fitting the scaler on the training data and transforming the validation and test data
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

In [ ]:
import pandas as pd

In [ ]:
housing_df = pd.DataFrame(housing.data, columns=housing.feature_names)
housing_df['target'] = housing.target
housing_df.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(housing_df['target'], bins=50, kde=True)
plt.title('Distribution of Target Variable')
plt.xlabel('Median House Value')
plt.ylabel('Frequency')
plt.savefig('target_distribution.png')
sns.pairplot(housing_df)
plt.savefig('pairplot.png')
plt.close()

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(housing_df.drop(columns=['target']),housing_df['target'],random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, random_state=42)

# Standardizing the data using StandardScaler
scaler = StandardScaler()

# Fitting the scaler on the training data and transforming the validation and test data
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)


In [ ]:
scaled_train_df = pd.DataFrame(X_train, columns=housing.feature_names)
scaled_train_df.head()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(scaled_train_df['MedInc'], bins=50, kde=True)
plt.title('Distribution of Median Income')
plt.xlabel('Median Income')
plt.ylabel('Frequency')

In [ ]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.Input(shape=(8,)), # Explicitly define Input layer
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(1)
])

model.compile(loss='mse', optimizer='adam')
history= model.fit(X_train, y_train, epochs=30, validation_data=(X_valid, y_valid))


In [ ]:
mes_test = model.evaluate(X_test, y_test)
print(mes_test)

In [ ]:
print('Making predictions on the test set:')
y_pred = model.predict(X_test)
print(y_pred[:5])
print('\nModel Summary:')
model.summary()

In [ ]:
import gradio as gr
import numpy as np

def predict_housing_price(MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude):
    # Combine inputs into a numpy array
    input_data = np.array([[MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude]])

    # Scale the input data using the pre-fitted scaler
    scaled_input_data = scaler.transform(input_data)

    # Make prediction using the trained model
    prediction = model.predict(scaled_input_data)[0][0]

    return prediction

# Create Gradio interface
iface = gr.Interface(
    fn=predict_housing_price,
    inputs=[
        gr.Number(label="Median Income (MedInc)", value=3.87),
        gr.Number(label="House Age (HouseAge)", value=29.0),
        gr.Number(label="Average Rooms (AveRooms)", value=6.24),
        gr.Number(label="Average Bedrooms (AveBedrms)", value=1.07),
        gr.Number(label="Population (Population)", value=1300.0),
        gr.Number(label="Average Occupancy (AveOccup)", value=3.09),
        gr.Number(label="Latitude (Latitude)", value=34.0),
        gr.Number(label="Longitude (Longitude)", value=-118.0)
    ],
    outputs=gr.Number(label="Predicted Median House Value"),
    title="California Housing Price Predictor",
    description="Enter the features of a house to predict its median value (in $100,000s)."
)

# Launch the interface
iface.launch()